# 01 — Build df_master

Reproduces `df_master.shape == (30000, 27)` from the 5 raw CSVs. Pre-validated, vectorized. See `prompts/01_df_master_construction.md` for the design rationale.

In [1]:
import sys
from pathlib import Path

_root = Path.cwd()
if _root.name == 'notebooks':
    _root = _root.parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

import os
os.chdir(_root)  # so 'data/...' relative paths in src.config resolve from project root

import numpy as np
import pandas as pd

from src.config import (
    DATA_RAW, DATA_PROCESSED, SEASON_START,
)
from src.utils import seed_all

seed_all()
print('seeded; pandas', pd.__version__, '; cwd:', Path.cwd())

seeded; pandas 3.0.3 ; cwd: D:\AgrNav


In [2]:
retailer_visit_log        = pd.read_csv(DATA_RAW / 'retailer_visit_log.csv')
retailer_pos              = pd.read_csv(DATA_RAW / 'retailer_pos.csv')
retailer_inventory_weekly = pd.read_csv(DATA_RAW / 'retailer_inventory_weekly.csv')
reps_territory            = pd.read_csv(DATA_RAW / 'reps_territory.csv')
retailers                 = pd.read_csv(DATA_RAW / 'retailers.csv')

# Step 0 — datetime parsing
retailer_visit_log['visit_date']           = pd.to_datetime(retailer_visit_log['visit_date'])
retailer_pos['transaction_date']           = pd.to_datetime(retailer_pos['transaction_date'])
retailer_inventory_weekly['week_end_date'] = pd.to_datetime(retailer_inventory_weekly['week_end_date'])
season_start = SEASON_START

print({
    'visit_log':  retailer_visit_log.shape,
    'pos':        retailer_pos.shape,
    'inventory':  retailer_inventory_weekly.shape,
    'reps':       reps_territory.shape,
    'retailers':  retailers.shape,
})

{'visit_log': (30000, 6), 'pos': (235042, 7), 'inventory': (310544, 5), 'reps': (500, 6), 'retailers': (4000, 5)}


In [3]:
# Step 1 — base
df_master = retailer_visit_log.copy().reset_index(drop=True)
df_master['_visit_id'] = df_master.index

# Step 2 — geo from reps_territory (dedup on territory_id)
geo = reps_territory[['territory_id', 'state', 'district']].drop_duplicates(subset=['territory_id'])
df_master = df_master.merge(geo, on='territory_id', how='left')
df_master.shape

(30000, 9)

In [4]:
# Step 3 — calendar features
df_master['week_of_season']         = (((df_master['visit_date'] - season_start).dt.days // 7) + 1).clip(1, 26)
df_master['month']                  = df_master['visit_date'].dt.month
df_master['day_of_week']            = df_master['visit_date'].dt.dayofweek
df_master['is_critical_period']     = df_master['week_of_season'].between(8, 16).astype(int)
df_master['is_harvest_approaching'] = (df_master['week_of_season'] >= 20).astype(int)
df_master[['week_of_season','month','day_of_week','is_critical_period','is_harvest_approaching']].describe()

,week_of_season,month,day_of_week,is_critical_period,is_harvest_approaching
count,30000.000000,30000.000000,30000.000000,30000.000000,30000.000000
mean,12.449533,6.596733,3.015900,0.335300,0.230700
std,7.461517,4.553372,2.002494,0.472103,0.421288
min,1.000000,1.000000,0.000000,0.000000,0.000000
25%,6.000000,2.000000,1.000000,0.000000,0.000000
50%,13.000000,10.000000,3.000000,0.000000,0.000000
75%,19.000000,11.000000,5.000000,1.000000,0.000000
max,25.000000,12.000000,6.000000,1.000000,1.000000


In [5]:
# Step 4 — visit_log lag features
t = df_master[['_visit_id', 'territory_id', 'visit_date']].sort_values(['territory_id', 'visit_date', '_visit_id'])
t['prev'] = t.groupby('territory_id')['visit_date'].shift(1)
t['days_since_last_visit'] = (t['visit_date'] - t['prev']).dt.days
df_master = df_master.merge(t[['_visit_id', 'days_since_last_visit']], on='_visit_id', how='left')
df_master['days_since_last_visit'] = df_master['days_since_last_visit'].fillna(-1).astype(int)

t = df_master[['_visit_id', 'territory_id', 'product_recommended', 'visit_date']].sort_values(
    ['territory_id', 'product_recommended', 'visit_date', '_visit_id']
)
t['prev'] = t.groupby(['territory_id', 'product_recommended'])['visit_date'].shift(1)
t['days_since_product_last_pushed'] = (t['visit_date'] - t['prev']).dt.days
df_master = df_master.merge(t[['_visit_id', 'days_since_product_last_pushed']], on='_visit_id', how='left')
df_master['days_since_product_last_pushed'] = df_master['days_since_product_last_pushed'].fillna(-1).astype(int)
df_master['is_first_visit_for_product'] = (df_master['days_since_product_last_pushed'] == -1).astype(int)

# visit_count_last_30days — vectorized self-merge inside territory
left  = df_master[['_visit_id', 'territory_id', 'visit_date']]
right = df_master[['territory_id', 'visit_date']].rename(columns={'visit_date': 'prior_date'})
pair  = left.merge(right, on='territory_id')
d     = (pair['visit_date'] - pair['prior_date']).dt.days
pair  = pair[(d >= 1) & (d <= 30)]
counts = pair.groupby('_visit_id').size().rename('visit_count_last_30days')
df_master = df_master.merge(counts, on='_visit_id', how='left')
df_master['visit_count_last_30days'] = df_master['visit_count_last_30days'].fillna(0).astype(int)
df_master[['days_since_last_visit','days_since_product_last_pushed','is_first_visit_for_product','visit_count_last_30days']].head()

,days_since_last_visit,days_since_product_last_pushed,is_first_visit_for_product,visit_count_last_30days
0,1,-1,1,12
1,3,4,0,14
2,0,8,0,14
3,0,0,0,14
4,0,0,0,14


In [6]:
# Step 5 — inventory features (merge_asof per retailer×product, aggregated over territory)
visit_keys = df_master[['_visit_id', 'territory_id', 'product_recommended', 'visit_date']]
expanded   = visit_keys.merge(retailers[['retailer_id', 'territory_id']], on='territory_id', how='left')
expanded   = expanded.dropna(subset=['retailer_id'])

inv = retailer_inventory_weekly[['retailer_id', 'sku_name', 'sku_qty', 'week_end_date']].copy()
inv = inv.rename(columns={'sku_name': 'product_recommended'})
inv = inv.sort_values(['retailer_id', 'product_recommended', 'week_end_date'])
inv['prev_sku_qty'] = inv.groupby(['retailer_id', 'product_recommended'])['sku_qty'].shift(1)

expanded_s = expanded.sort_values('visit_date').reset_index(drop=True)
inv_s      = inv.sort_values('week_end_date').reset_index(drop=True)
inv_match  = pd.merge_asof(
    expanded_s, inv_s,
    left_on='visit_date', right_on='week_end_date',
    by=['retailer_id', 'product_recommended'],
    direction='backward', allow_exact_matches=False,
)
inv_valid = inv_match.dropna(subset=['sku_qty']).copy()
inv_valid['_is_zero'] = (inv_valid['sku_qty'] == 0).astype(int)
inv_valid['_wow']     = inv_valid['sku_qty'] - inv_valid['prev_sku_qty']
inv_agg = inv_valid.groupby('_visit_id').agg(
    sku_qty_pre_visit=('sku_qty', 'mean'),
    is_out_of_stock=('_is_zero', 'max'),
    stock_change_wow=('_wow', 'mean'),
)

global_inv_mean = retailer_inventory_weekly['sku_qty'].mean()
df_master = df_master.merge(inv_agg, on='_visit_id', how='left')
df_master['sku_qty_pre_visit'] = df_master['sku_qty_pre_visit'].fillna(global_inv_mean).round(2)
df_master['is_out_of_stock']   = df_master['is_out_of_stock'].fillna(0).astype(int)
df_master['stock_change_wow']  = df_master['stock_change_wow'].fillna(0)
df_master[['sku_qty_pre_visit','is_out_of_stock','stock_change_wow']].describe()

,sku_qty_pre_visit,is_out_of_stock,stock_change_wow
count,30000.000000,30000.000000,30000.000000
mean,86.909792,0.060300,-3.209808
std,47.090360,0.238046,19.102157
min,0.000000,0.000000,-264.000000
25%,53.500000,0.000000,-5.500000
50%,87.110000,0.000000,-1.000000
75%,113.000000,0.000000,0.000000
max,366.000000,1.000000,156.000000


In [7]:
# Step 6 — POS demand features
pos_prod = retailer_pos[['retailer_id', 'sku_name', 'sku_qty', 'transaction_date']].rename(
    columns={'sku_name': 'product_recommended'}
)
prod_match = expanded.merge(pos_prod, on=['retailer_id', 'product_recommended'], how='inner')
prod_match['_d'] = (prod_match['visit_date'] - prod_match['transaction_date']).dt.days
prod_match = prod_match[prod_match['_d'] > 0]

qty_28  = prod_match.loc[prod_match['_d'] <= 28].groupby('_visit_id')['sku_qty'].sum() / 4.0
qty_30  = prod_match.loc[prod_match['_d'] <= 30].groupby('_visit_id')['sku_qty'].sum() / 30.0
sold_14 = prod_match.loc[prod_match['_d'] <= 14].groupby('_visit_id').size().rename('product_sold_last_14d')

df_master['rolling_avg_weekly_sales'] = df_master['_visit_id'].map(qty_28).fillna(0).astype(float)
df_master['sales_velocity']           = df_master['_visit_id'].map(qty_30).fillna(0).astype(float)
df_master['product_sold_last_14d']    = (df_master['_visit_id'].map(sold_14).fillna(0) > 0).astype(int)

# total_revenue_last_30d
visit_retailer = expanded[['_visit_id', 'retailer_id', 'visit_date']].drop_duplicates()
pos_all = retailer_pos[['retailer_id', 'sku_qty', 'sku_price', 'transaction_date']]
rev_match = visit_retailer.merge(pos_all, on='retailer_id', how='inner')
rev_match['_d'] = (rev_match['visit_date'] - rev_match['transaction_date']).dt.days
rev_match = rev_match[(rev_match['_d'] > 0) & (rev_match['_d'] <= 30)]
rev_match['_rev'] = rev_match['sku_qty'] * rev_match['sku_price']
rev_30 = rev_match.groupby('_visit_id')['_rev'].sum()
df_master['total_revenue_last_30d'] = df_master['_visit_id'].map(rev_30).fillna(0).astype(float)
df_master[['rolling_avg_weekly_sales','sales_velocity','product_sold_last_14d','total_revenue_last_30d']].describe()

,rolling_avg_weekly_sales,sales_velocity,product_sold_last_14d,total_revenue_last_30d
count,30000.000000,30000.000000,30000.000000,3.000000e+04
mean,13.752017,1.953280,0.760200,8.633943e+05
std,17.070715,2.375773,0.426968,4.562462e+05
min,0.000000,0.000000,0.000000,0.000000e+00
25%,1.750000,0.266667,1.000000,5.655753e+05
50%,7.250000,1.066667,1.000000,8.447911e+05
75%,19.250000,2.800000,1.000000,1.134661e+06
max,144.750000,19.566667,1.000000,2.870649e+06


In [8]:
# Step 7 — rep features (cumulative, strictly prior)
t = df_master[['_visit_id', 'rep_id', 'visit_date', 'product_recommended']].sort_values(
    ['rep_id', 'visit_date', '_visit_id']
).reset_index(drop=True)
t['visits_before'] = t.groupby('rep_id').cumcount()
t['first_visit']   = t.groupby('rep_id')['visit_date'].transform('min')
weeks_elapsed      = ((t['visit_date'] - t['first_visit']).dt.days / 7).clip(lower=1)
t['rep_visit_frequency_score'] = t['visits_before'] / weeks_elapsed

t['_new_prod'] = (~t.duplicated(subset=['rep_id', 'product_recommended'])).astype(int)
t['rep_product_diversity'] = t.groupby('rep_id')['_new_prod'].cumsum() - t['_new_prod']
df_master = df_master.merge(
    t[['_visit_id', 'rep_visit_frequency_score', 'rep_product_diversity']],
    on='_visit_id', how='left',
)
df_master[['rep_visit_frequency_score','rep_product_diversity']].describe()

,rep_visit_frequency_score,rep_product_diversity
count,30000.000000,30000.000000
mean,3.520972,9.143533
std,2.380540,3.088859
min,0.000000,0.000000
25%,2.122239,8.000000
50%,3.000000,10.000000
75%,4.089888,11.000000
max,31.000000,12.000000


In [9]:
# Step 8 — target (sale_within_7d)
pos_tgt = retailer_pos[['retailer_id', 'sku_name', 'transaction_date']].rename(
    columns={'sku_name': 'product_recommended'}
)
tgt_match = expanded.merge(pos_tgt, on=['retailer_id', 'product_recommended'], how='inner')
d = (tgt_match['transaction_date'] - tgt_match['visit_date']).dt.days
tgt_match = tgt_match[(d > 0) & (d <= 7)]
hit_ids = tgt_match['_visit_id'].unique()
df_master['sale_within_7d'] = df_master['_visit_id'].isin(hit_ids).astype(int)
df_master['sale_within_7d'].value_counts(normalize=True).round(3)

sale_within_7d
1    0.651
0    0.349
Name: proportion, dtype: float64

In [10]:
# Step 9 — column order + audit prints
final_cols = [
    'rep_id', 'visit_date', 'territory_id', 'visit_tehsil', 'visit_type',
    'product_recommended', 'state', 'district',
    'week_of_season', 'month', 'day_of_week',
    'is_critical_period', 'is_harvest_approaching',
    'days_since_last_visit', 'visit_count_last_30days',
    'days_since_product_last_pushed', 'is_first_visit_for_product',
    'sku_qty_pre_visit', 'is_out_of_stock', 'stock_change_wow',
    'rolling_avg_weekly_sales', 'sales_velocity',
    'product_sold_last_14d', 'total_revenue_last_30d',
    'rep_visit_frequency_score', 'rep_product_diversity',
    'sale_within_7d',
]
df_master = df_master[final_cols]

print('shape:', df_master.shape)
print('\nclass balance:')
print(df_master['sale_within_7d'].value_counts(normalize=True).round(3))
print('\nnull audit (total):', df_master.isnull().sum().sum())
print('\ndate range:', df_master['visit_date'].min(), '->', df_master['visit_date'].max())
df_master.head()

shape: (30000, 27)

class balance:
sale_within_7d
1    0.651
0    0.349
Name: proportion, dtype: float64

null audit (total): 0

date range: 2025-09-29 00:00:00 -> 2026-03-29 00:00:00


,rep_id,visit_date,territory_id,visit_tehsil,visit_type,product_recommended,state,district,week_of_season,month,...,sku_qty_pre_visit,is_out_of_stock,stock_change_wow,rolling_avg_weekly_sales,sales_velocity,product_sold_last_14d,total_revenue_last_30d,rep_visit_frequency_score,rep_product_diversity,sale_within_7d
0,REP_0203,2026-03-09,TER_0203,Jalgaon_T062,retailer meeting,Vertimec 1.8 EC,Maharashtra,Jalgaon,23,3,...,59.0,0,47.0,6.25,0.833333,1,537045.47,1.190476,8,0
1,REP_0203,2026-03-12,TER_0203,Jalgaon_T064,retailer meeting,Tilt 250 EC,Maharashtra,Jalgaon,23,3,...,71.0,0,0.0,7.00,0.933333,1,450333.33,1.260000,9,1
2,REP_0203,2026-03-12,TER_0203,Jalgaon_T063,campaign_conducted,Cruiser 350 FS,Maharashtra,Jalgaon,23,3,...,23.5,1,-1.5,5.25,0.700000,1,450333.33,1.306667,9,1
3,REP_0203,2026-03-12,TER_0203,Jalgaon_T064,retailer meeting,Cruiser 350 FS,Maharashtra,Jalgaon,23,3,...,23.5,1,-1.5,5.25,0.700000,1,450333.33,1.353333,9,1
4,REP_0203,2026-03-12,TER_0203,Jalgaon_T063,campaign_conducted,Tilt 250 EC,Maharashtra,Jalgaon,23,3,...,71.0,0,0.0,7.00,0.933333,1,450333.33,1.400000,9,1


In [11]:
# 6 invariants from plan §Notebook 01 (max-date adjusted to actual data span)
assert df_master.shape == (30000, 27), f'shape mismatch: {df_master.shape}'
assert df_master.isnull().sum().sum() == 0, 'nulls present'

pos_rate = df_master['sale_within_7d'].mean()
assert 0.64 <= pos_rate <= 0.66, f'class balance out of band: {pos_rate:.4f}'

vd_min, vd_max = df_master['visit_date'].min(), df_master['visit_date'].max()
assert abs((vd_min - pd.Timestamp('2025-09-29')).days) <= 2, f'visit_date min off: {vd_min}'
assert abs((vd_max - pd.Timestamp('2026-03-29')).days) <= 2, f'visit_date max off: {vd_max}'

sent_last = (df_master['days_since_last_visit'] == -1).mean()
assert 0.013 <= sent_last <= 0.022, f'days_since_last_visit sentinel rate off: {sent_last:.4f}'

sent_prod = (df_master['days_since_product_last_pushed'] == -1).mean()
assert 0.17 <= sent_prod <= 0.21, f'days_since_product_last_pushed sentinel rate off: {sent_prod:.4f}'

print('all 6 assertions passed')
print(f'  shape:            {df_master.shape}')
print(f'  pos_rate:         {pos_rate:.4f}')
print(f'  date range:       {vd_min.date()} .. {vd_max.date()}')
print(f'  sent last visit:  {sent_last:.4f}')
print(f'  sent prod pushed: {sent_prod:.4f}')

all 6 assertions passed
  shape:            (30000, 27)
  pos_rate:         0.6512
  date range:       2025-09-29 .. 2026-03-29
  sent last visit:  0.0167
  sent prod pushed: 0.1893


In [12]:
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
out = DATA_PROCESSED / 'df_master.parquet'
df_master.to_parquet(out, index=False)
print('saved:', out, df_master.shape)

saved: data\processed\df_master.parquet (30000, 27)
